In [ ]:
# ==============================================================================
# eHRAF OUTLINE OF CULTURAL MATERIALS (OCM) INGESTION: STEP 1
# 
# ARCHITECTURE NOTE: eHRAF OCM is an ethnographic subject classification.
# This script uses Strategy B (Targeted Web Scraping) via BeautifulSoup to 
# crawl the live web application.
#
# SSSOM ALIGNMENT STRATEGY: 
# 1. Hierarchy: Dynamically builds absolute paths by climbing "Broader Term(s)".
# 2. Scope Constraints: Recursively crawls downward ("Narrower Terms"), but 
#    intentionally drops "Related Terms" to maintain taxonomic boundaries.
# 3. Synonyms: Left blank during Step 1; populated offline in Step 3.
# ==============================================================================

import os
import sys
import re
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py. Check PYTHONPATH.")

# --- 2. Registry Lookup ---
SOURCE_PREFIX = "OCM"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="Outline of Cultural Materials",
    fallback_uri="https://ehrafworldcultures.yale.edu/subjects/"
)
SOURCE_NAME = registry_data['Source_Name']
BASE_URI = registry_data['Base_URI']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")

HEADERS = {'User-Agent': 'ReligiousMappingProject/1.0', 'Accept': 'text/html'}
SESSION = requests.Session()

TARGET_SEEDS = [
                "770", # religious beliefs
                "780",  # religious practices
                # --- Additional seeds from lateral discovery ---
                "760", # death
                "790", # ecclesiastical organization
                "688", # religious offenses
                "755", # magical and mental therapy
                "754", # sorcery
                # --- Additional seeds from lateral discovery round 2 ---
                "756", # shamans and psychotherapists
                "346", # religious and educational structures
               ]

# --- 3. Persistent Tracking ---
if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch. Deleting {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].tolist())
        print(f"[*] Resuming: Found {len(processed_ids)} OCM concepts already saved.")
else:
    processed_ids = set()

path_cache = {}

# --- 4. Helper Functions ---
def fetch_html_with_retry(url):
    for attempt in range(3):
        try:
            res = SESSION.get(url, headers=HEADERS, timeout=10)
            if res.status_code == 200:
                return res.text
            elif res.status_code in [429, 500, 502, 503, 504]:
                print(f"    [!] Server error {res.status_code}. Retrying in {2 ** attempt}s...")
            else:
                print(f"    [!] HTTP {res.status_code} for {url}. Skipping.")
                return None
        except requests.exceptions.RequestException as e:
            print(f"    [!] Request failed: {e}. Retrying in {2 ** attempt}s...")
        time.sleep(2 ** attempt)
    return None

def extract_list_links(soup, header_keyword):
    """
    Locates an h6 header using regex to handle singular/plural variations 
    and extracts IDs and labels from the immediate ul sibling.
    """
    header = soup.find('h6', string=re.compile(header_keyword, re.I))
    if not header: return []
    
    ul = header.find_next_sibling('ul')
    if not ul: return []
    
    results = []
    for a in ul.find_all('a'):
        match = re.search(r'subjects/(\d+)', a.get('href', ''))
        if match:
            # Strip the leading numeric ID from the label text
            label = re.sub(r'^\d+\s*', '', a.text).strip()
            results.append({'id': match.group(1), 'label': label})
    return results

def get_ocm_path(cid, current_label):
    if cid in path_cache: return path_cache[cid]
    
    html = fetch_html_with_retry(f"{BASE_URI}{cid}/")
    if not html: return current_label

    soup = BeautifulSoup(html, 'html.parser')
    broader_terms = extract_list_links(soup, "broader term")
    
    if not broader_terms:
        path_cache[cid] = current_label
        return current_label
    
    primary_parent = broader_terms[0]
    parent_path = get_ocm_path(primary_parent['id'], primary_parent['label'])
    
    full_path = f"{parent_path} > {current_label}"
    path_cache[cid] = full_path
    return full_path

def process_node(cid):
    if cid in processed_ids: return

    url = f"{BASE_URI}{cid}/"
    html = fetch_html_with_retry(url)
    if not html: return

    soup = BeautifulSoup(html, 'html.parser')
    
    # Primary Label
    h1 = soup.find('h1')
    if not h1: return
    primary_label = h1.text.strip()
    
    print(f"\r[Ingesting] {primary_label[:40]:<40} (ID: {cid})", end="", flush=True)

    # Scope Notes (Description)
    description = ""
    scope_header = soup.find('h6', string=re.compile("Scope Note", re.I))
    if scope_header and scope_header.find_next_sibling('p'):
        description = re.sub(r'\s+', ' ', scope_header.find_next_sibling('p').text).strip()

    # Parent Resolution
    broader_terms = extract_list_links(soup, "broader term")
    parent_ids = " | ".join([p['id'] for p in broader_terms])

    # Write to standard schema
    extracted_data = {
        'Source_System': SOURCE_NAME, 
        'Primary_Label': primary_label,
        'CURIE': f"{SOURCE_PREFIX}:{cid}", 
        'Formal_Label': "", 
        'Concept_Type': 'skos:Concept',
        'Hierarchy_Path': get_ocm_path(cid, primary_label), 
        'Synonyms': "", 
        'Description': description, 
        'Parent_IDs': parent_ids,
        'Concept_ID': str(cid), 
        'URI': url, 
        'Has_Translation': "",
        'Status': 'active',
        'Crosswalks': "" 
    }
    
    clean_row = finalize_row(extracted_data)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )
    processed_ids.add(cid)

    # Recurse Downward into Narrower Terms
    narrower_terms = extract_list_links(soup, "narrower term")
    for child in narrower_terms:
        time.sleep(0.5) 
        process_node(child['id'])

# --- 5. Execution ---
print(f"Starting {SOURCE_PREFIX} Primary Ingestion...\n")
for seed_id in TARGET_SEEDS:
    process_node(seed_id)

print(f"\n\n[COMPLETE] Extracted {len(processed_ids)} OCM records.")

In [ ]:
# ==============================================================================
# eHRAF OUTLINE OF CULTURAL MATERIALS (OCM): STEP 2 LATERAL DISCOVERY
# Run this cell to harvest "Related Terms" outside the current extracted scope.
# ==============================================================================

import os
import re
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RUN_LATERAL_DISCOVERY = True

if RUN_LATERAL_DISCOVERY:
    # --- 1. Setup Paths & Environment ---
    notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
    raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
    
    raw_ocm_file = os.path.join(raw_data_dir, "raw_ocm.csv")
    candidates_file = os.path.join(raw_data_dir, "ocm_lateral_candidates.csv")
    
    BASE_URI = "https://ehrafworldcultures.yale.edu/subjects/"
    HEADERS = {'User-Agent': 'ReligiousMappingProject/1.0', 'Accept': 'text/html'}
    SESSION = requests.Session()

    # --- 2. Load Existing Concepts (Deduplication Baseline) ---
    if not os.path.exists(raw_ocm_file):
        raise FileNotFoundError("Please run Cell 1 to generate raw_ocm.csv first.")
        
    existing_df = pd.read_csv(raw_ocm_file, dtype={'Concept_ID': str})
    processed_ids = set(existing_df['Concept_ID'].tolist())
    print(f"[*] Loaded {len(processed_ids)} existing OCM concepts for deduplication.")

    known_candidates = set()
    if os.path.exists(candidates_file):
        cand_df = pd.read_csv(candidates_file, dtype={'Candidate_ID': str})
        if 'Candidate_ID' in cand_df.columns:
            known_candidates = set(cand_df['Candidate_ID'].astype(str).tolist())
            print(f"[*] Loaded {len(known_candidates)} previous candidates from Inbox.")

    # --- 3. Helper Functions ---
    path_cache = {}

    def fetch_html_with_retry(url):
        for attempt in range(3):
            try:
                res = SESSION.get(url, headers=HEADERS, timeout=10)
                if res.status_code == 200: return res.text
            except Exception: pass
            time.sleep(2 ** attempt)
        return None

    def extract_list_links(soup, header_keyword):
        header = soup.find('h6', string=re.compile(header_keyword, re.I))
        if not header: return []
        
        ul = header.find_next_sibling('ul')
        if not ul: return []
        
        results = []
        for a in ul.find_all('a'):
            match = re.search(r'subjects/(\d+)', a.get('href', ''))
            if match:
                label = re.sub(r'^\d+\s*', '', a.text).strip()
                results.append({'id': match.group(1), 'label': label})
        return results

    def get_ocm_path(cid, current_label):
        if cid in path_cache: return path_cache[cid]
        html = fetch_html_with_retry(f"{BASE_URI}{cid}/")
        if not html: return current_label
        
        soup = BeautifulSoup(html, 'html.parser')
        broader_terms = extract_list_links(soup, "broader term")
        
        if not broader_terms:
            path_cache[cid] = current_label
            return current_label
            
        primary_parent = broader_terms[0]
        parent_path = get_ocm_path(primary_parent['id'], primary_parent['label'])
        full_path = f"{parent_path} > {current_label}"
        path_cache[cid] = full_path
        return full_path

    # --- 4. Harvest Related Concepts ---
    new_candidates = []
    
    print("\nScanning extracted concepts for lateral associations...\n")
    for i, cid in enumerate(list(processed_ids), 1):
        print(f"\r[{i}/{len(processed_ids)}] Checking relations for ID: {cid:<15}", end="", flush=True)
        
        html = fetch_html_with_retry(f"{BASE_URI}{cid}/")
        if not html: continue
            
        soup = BeautifulSoup(html, 'html.parser')
        
        # Regex catches "Related Term" or "Related Terms"
        related_terms = extract_list_links(soup, "related term")
        
        for r_node in related_terms:
            r_id = r_node['id']
            r_label = r_node['label']
            
            # Deduplicate against extracted data AND previous inbox items
            if r_id not in processed_ids and r_id not in known_candidates:
                if not any(d['Candidate_ID'] == r_id for d in new_candidates):
                    # Fetch the full hierarchy path for context
                    context_path = get_ocm_path(r_id, r_label)
                    
                    new_candidates.append({
                        'Candidate_ID': r_id, 
                        'Candidate_Label': r_label,
                        'Hierarchy_Path': context_path,
                        'Source_Parent_ID': cid
                    })
        time.sleep(0.5) # Polite delay

    # --- 5. Export and Sort ---
    print(f"\n\nDiscovery Complete! Found {len(new_candidates)} net-new candidates.")
    
    if new_candidates:
        new_df = pd.DataFrame(new_candidates)
        # Sort by hierarchy so entire branches group together visually
        new_df = new_df.sort_values(by='Hierarchy_Path')
        new_df.to_csv(candidates_file, mode='a', index=False, 
                      header=not os.path.isfile(candidates_file), encoding='utf-8-sig')
        
        print(f"Sorted candidates appended to: {candidates_file}")
        print("Review the CSV and add any relevant branches to your TARGET_SEEDS in Cell 1.")
    else:
        print("No new relevant concepts found. Your seed list is comprehensive!")
else:
    print("Lateral Discovery is disabled. Set RUN_LATERAL_DISCOVERY = True to execute.")

In [ ]:
# ==============================================================================
# eHRAF OUTLINE OF CULTURAL MATERIALS (OCM): STEP 3 SYNONYM ENRICHMENT
# Run this cell after the Bronze Layer is finalized. It parses the local 
# ocm-2025.pdf to extract "Use Fors:" terms and injects them as Synonyms.
# ==============================================================================

import os
import re
import pandas as pd
import pdfplumber

# --- 1. Environment Setup ---
notebook_dir = os.path.dirname(os.path.abspath("__file__")) if '__file__' in locals() else os.getcwd()
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
external_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "external"))

raw_ocm_file = os.path.join(raw_data_dir, "raw_ocm.csv")
pdf_path = os.path.join(external_data_dir, "ocm-2025.pdf")

if not os.path.exists(raw_ocm_file):
    raise FileNotFoundError(f"Could not find {raw_ocm_file}. Please run Steps 1 and 2 first.")
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF not found at {pdf_path}. Please download it.")

# --- 2. Helper Functions ---
def fix_ligatures(text):
    """
    Replaces standard PDF ligatures and specific Private Use Area (PUA) 
    characters resulting from broken PDF encoding mappings.
    """
    replacements = {
        '\ufb00': 'ff',
        '\ufb01': 'fi',
        '\ufb02': 'fl',
        '\ufb03': 'ffi',
        '\ufb04': 'ffl',
        '': 'ffi', # eHRAF specific PUA char
        '': 'ff',  # eHRAF specific PUA char
        '': 'fi',  # eHRAF specific PUA char (e.g., Autosacrifice)
        '': 'fl',  # eHRAF specific PUA char (e.g., Genuflection)
        'ﬁ': 'fi',
        'ﬂ': 'fl',
        'ﬀ': 'ff'
    }
    for bad_char, good_chars in replacements.items():
        text = text.replace(bad_char, good_chars)
    return text

# --- 3. Load Target Concept IDs (Strict String Typing) ---
print(f"[*] Loading extracted OCM concepts...")
# Force ALL columns to string to prevent Pandas from converting Parent_IDs to floats
df = pd.read_csv(raw_ocm_file, dtype=str) 

# Ensure 'Synonyms' column exists and replace NaNs with empty strings
if 'Synonyms' not in df.columns:
    df['Synonyms'] = ""
df.fillna("", inplace=True)

target_ids = set(df['Concept_ID'].str.strip().tolist())
print(f"[*] Targeting {len(target_ids)} concepts for synonym enrichment.")

# --- 4. Parse PDF with State Machine ---
print(f"[*] Scanning {os.path.basename(pdf_path)} for synonyms... (This will take a minute)")

synonym_map = {cid: [] for cid in target_ids if cid}
current_id = None
capturing_synonyms = False

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if not text: continue
        
        # Apply ligature fixes to the entire page before splitting lines
        text = fix_ligatures(text)
        
        for line in text.split('\n'):
            line = line.strip()
            if not line: continue
            
            # Detect a major subject header (e.g., "782 PRAYERS AND SACRIFICES")
            header_match = re.match(r'^(\d{3})\s+[A-Z\s&,-]+$', line)
            if header_match:
                current_id = header_match.group(1)
                capturing_synonyms = False # Reset capture state on new section
                continue
            
            # Optimization: Skip lines if we aren't in a target concept block
            if current_id not in target_ids:
                continue
                
            # Detect the "Use Fors:" block
            if re.match(r'^Use\s*Fors?:?$', line, re.IGNORECASE):
                capturing_synonyms = True
                continue
                
            # Capture the synonyms
            if capturing_synonyms:
                # Stop conditions: Next major metadata block
                if re.match(r'^(History Note|Broader Terms?|Related Terms?|Scope Note):$', line, re.IGNORECASE):
                    capturing_synonyms = False
                    continue
                    
                # Skip pagination footers (e.g., "617 | Outline of Cultural Materials")
                if re.search(r'^\d+\s*\|\s*Outline of Cultural Materials', line, re.IGNORECASE):
                    continue
                # Fallback skip for isolated page numbers just in case
                if re.match(r'^\d+$', line): 
                    continue
                    
                # Clean up and append the synonym
                if len(line) > 1: # Ignore stray single characters
                    synonym_map[current_id].append(line)

# --- 5. Inject Synonyms into DataFrame ---
updated_count = 0
for index, row in df.iterrows():
    cid = row['Concept_ID'].strip()
    if cid in synonym_map and len(synonym_map[cid]) > 0:
        # Deduplicate and format as pipe-separated string
        unique_syns = list(dict.fromkeys(synonym_map[cid])) 
        df.at[index, 'Synonyms'] = " | ".join(unique_syns)
        updated_count += 1

# Save back to CSV, preserving exact formatting
df.to_csv(raw_ocm_file, index=False, encoding='utf-8-sig')
print(f"\n[COMPLETE] Successfully enriched {updated_count} OCM concepts with 'Use For' synonyms.")